# Curva de aprendizaje: ¿más imágenes suben la nota?

Decidido con jhonatan el 21 oct 2026 (plan en `docs/experimentos.md`, "Curva de aprendizaje"). Mismo sistema que la
ronda 5 (YOLO11m-seg a 1024 px + refinador de la ronda 3) y mismo examen de 121 imágenes; solo cambia cuántas
imágenes de estudio ve YOLO: 25 %, 50 % y 75 % (cada grupo contiene al anterior). El 100 % ya está medido en la
ronda 5: **PQ 0,4532** (umbral 0,2; TP 922 · FP 481 · FN 445).

Requisitos en Kaggle: GPU T4, **Internet On** y como *Input* la competición y `filamentos-detector-ronda-2`
Version 4 (`refinador.pt`). No genera envío.

In [ ]:
# Configuración. PRUEBA_RAPIDA=True reduce todo para comprobar que el notebook funciona.
PRUEBA_RAPIDA = False

FRACCIONES = (0.25, 0.5, 0.75)  # parte de las imágenes de estudio que ve YOLO en cada paso
RES = 1024
MODELO = 'yolo11m-seg.pt'
HORAS = 1.75               # tope de entrenamiento por paso (el 100 % ya no mejoraba desde ~1,7 h)
EPOCAS = 300
PACIENCIA = 25
LOTE = 6
CONF_MIN = 0.1
UMBRAL_REF = 0.5
UMBRALES = (0.15, 0.2, 0.25, 0.3, 0.4, 0.5)
SEMILLA = 0

In [ ]:
import subprocess, sys
for paquete, modulo in (('ultralytics', 'ultralytics'), ('segmentation-models-pytorch', 'segmentation_models_pytorch'),
                        ('pycocotools', 'pycocotools')):
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete], check=True)

## Código del proyecto
Empaquetado del repositorio `filamentos-solares-2026` (los mismos ficheros de `filamentos/`).

In [ ]:
import base64, io, os, tarfile
tarfile.open(fileobj=io.BytesIO(base64.b64decode('H4sIAPMP2WoC/+19W5PbRpamn+tXZNCxbcBGQbyWSnTT0RpJ7vGsWtbI9k5ssBkMFJhkoQoEYAAsk9LK/2B+RD/2gx8m5m1e64/NuWQmEiBYVXK7vdtrIqJYJJDXk5knz+XLA/+R/+gPr4PtP8tgIfOP/i5Xl69D/7vdwaD6jvd73X6v95HYfvQrXJuiDHKo/qPf5tV/LNZltJaT3uMn/cFwODo797vD7uBx/+Sj4/X//7WM4mAtkzItHs3nURKV87mf7X759X82HB5c/73u4KPeqH826ne7jx/j+h9C+o9E99dc/3malnelu+/5P+jV6XSe3f7nIlqlYiFjkeXpToYl/hCFXOHMCMLo9j8TvFHNFVGkcZDLwofsRz7xj3z5/0/s/8P9/b9/3P9/lf3/vLb/n533e/7jJ73RaDg6Luzf1v6/CODzF9/879//h8O+2f9Hg+4ZrP/BcNQ97v+/0v7/Ejb8TR7gDv+np3/8Mnr5tdiJME1uZF7Q1g/TI5ciS+Pbn1ZpkhaeWN/+pQiDPCgg5ZuXL0gMiNZZmpfiqkgT/T0tTpZ5uobC4hgqidKkEOrRQi6DTVwuorA80cmTzTrbCSg0yThftgvTMC3TNDb51kFxjUnw/3xTRnFxcvLt0z+Jieh3h+cnJydQsICmrYJ8HiQpSi9pIgsnD6K37vhEwAWNfS5vNjK+keJdkq4vcjmP1sFKJmPxjvIs0nwspio7UuDZ18++9oTv+7P37/0TKuW7JBCcS2QbCbQrZSJzcRnAfBIDkctVVJQ5EmuTpEC8XOiiffEiFtGC5C2djIqUhej8Ea6nT09/z+36ovO52BTBGiQufgCkXKeQGYYkWkYhFkflmLJ1D+n/D1F5KdJMJg4u7KC89K/SKCFaeKJTwv+kA1/UqM97fnf+NMGSaKjm18FqFct+t382p7Q+jm3HdZH+S6YlXjhGQH986MdpsHCWLj3TfYNn76JpJ1p0ZmPhwDfgOXKeANvpzDyhnvhFFkel0zntuNPuzBVL6FgkooRKhyRI6aIze08lp5sSCrWmkANM7GIRjOv3oHaXm4KlQXPm0cITDpPWI5q5WIVuqB+Vcl04btU1qGjKyWc0HWZCfAyjfvsXGHYYLaY6UKOAUizpuITWRLd/hcZvUIBOTBuCqkdBRWigi6nRbhz0UTdtGigazJFYs8MN9IMMxnvhBHoMYGkn4t01kAZpcsOUvfbEDTYFsutOv1drB5Z5hKt8HsxhjeEidypFIE2gZXGZTmDJYRvDS/qqKJbHQJOJtTT9Zf569/XFFaz9oqUUVUKtpVbmhQzThXSsO2uZr6SD1biuDwttl0nnAriDq9quGlzMIZu1+HXzuIZpSw+Burp5OCIdNQFpuKyCZvV6oABoi6N+VdzlGYjSCfHTT8J0k5TFJ8gk8TfzEVivUPo6YDUrDpA/btaQw24DFu3rhQzf64SVCdEmyYAMUBYszyTI82CnG6OpAwk2UVKeu26NylDetMNtg6WnCN2BrFHUcfc6qQlW6NI9UaaxhDrDKJgA17C6nsICyCEp9mwDXdI7BTMuXJKwXKbbrid28Lftwf8ec1bhaKIglVxPZJgNaHUTFcDYNCloY4CGwk6ht40FrQzaOBaRleia7+udQwbFJpfMvXfAl7c4WSGHv4ySxTxV0/QgAYEtUdY4DYMYc2bAUhc6/ZRLBIbWY0obokHS6cxwgBCnlGoKVxwiyTZ54VDBe/V6ouuPLJYU4kRQ+WGt5+kWellKHKQd1OeE9uC4Jlu0FDHsA6ErvpiIQVVcramaeUxzmBrAx4GbI8vAPlHjd9BF4mD0iziIsxWfQb99lKJKcSpwQOHOzrrjzmpzz9Sm5hnMRWsl8qS8k8scYhTvOkX0VnZw566YCwyInuhjwV/04tHz/X2Dlxz1/6P+/1vV/5X9HwZg8Pio//+29P8wDoooTH95C8A9+v9Zt9tr2P8Hj4ejo/7/a+n/tz8lMhAXQSFFGIO8BnrlWKQXhcxvSMF03qYJCFdpEW5AuHZhe7/ayLfCAeUkxvsqJTwC3fgbFJmzHMSI6G1wBVLZpkxBCCxxZolvohyUbhbsQK0BKTlvSMMyubn9KSUDxDo9yeVSQiqQZNAgAVIuSsvfbyRIQIsoB1GsEGvYyuG/f4ICIsl9iwDkdWhYIStrg7512Nbw+quXOvVXKDae3Cto3iFketCsPLtM43S1AzHnD1X99CleAwXWEs0DLNNsQIML4jlROB0LErxAzuv65+f4GHTOpJRJES2CBcg/MSgINwGISEC4ZJGyRMqyUbCIUpCltnYZT7pa0PtY0JCJ/lgkMF4knEMxMOZkRICE/4OtElgMFRjkEmSzKBljC6C0waBrS466wMEYdd7g9j9SKPOnJFpzbuoPNpqLqJo0HNq5h2PoAjcmkas8FUWK8h2Ib4HxNHFrYjTsrCPkV/Uie/55Vd5ozDNKpV8ETOMEtapsq7tyJkS9KzAcb2FCAVmkUmAk2nIWqLPQFCxQpZFoD4P/lb6v5FhIF6Zzaq4TrVeWToT2M5zSRFYiMKWFmmAllFFOWiDUmN3+tJWoQ1/kURwHMOSFUXvW0GIoVXwB1BuIT0n7kHmIdiBQQeGJJ548cU1SVGsuIlheuzm0M57DVJSFY91Fq1CUrJy1J6IStAUyREwGSk2MgwtPzFU58EPGzpqfXLPmA3SFWYZaCpRHYrUDyfw8uIGkoCv1xjPkEz1WtlBv2HLGJE3eyjzF1GIyEdc11WC7BRU/SBzQOHY78xVyFd/nUAHoL9utKx5R5yOtpyY41DEwm5zJEG7hD2p8Y5n7optoIckIJ2uLRjjISEB3xjG9IVtNQCP/Y6/bFdnWNfS3+wBaWxQCOaE6v7gMMqk6wU8vd1laQkNB9cGm7Hb4ZYfNfkPJFjwdJpD+94pAsKShcynn/+ESWJ7DqTxBXYK73E6scsqPZmqoMon8kAelYBsA8S8f2RP/VsVbqiUtm0HfdX1YWqA0Oc6o1/cEfAC9Ofs/ffXyq1cvnr4xFjQkGk8IbgtOLJg5DjXAE1jMpD9qpD7cKEpi6jeknI7Hp73ZgUbgUKt1QJMApmC03qy5LKN5Y6rpj4pKyBvsKYYPPZGruVNtco564InMNuRAd8ktMha1PVAPolnLaOUFtgSdVlxkvYEtDBZogGq1nkTM36FJVBsMPyj5Nud3xe9gV6X7hpO79ZwtS5if1daxosTH4iVaI7lFujXIxoJVvskCNEUmZKVZAC9Ttm4gAKxP3GeN6WYJ9cVonYabaR6t8CeVj6WQgUMbJJhVWG2kgqFJppGw62/Q3SAn1Q7pAwmvHaCF4tANY5Wq5VNFBDV0KII4soxg8pVB4amBtQbvaRaDICNQQlGCCggRuOuFelSBJS+0IwCfV16NIJQZWXbN0Jk7dXMOlGwbdNBcmyZZnmZF1TbLeBMtMYePuyqNstlea/YYFHWiZCPNTTKVddlUtu2xWdi/uEi3JsVNEJM1C6f+rjumhN3xtjebYlISUWZ2KzC9DxU7LrWjvlHf3Rjdh21UwPy8SvM5MJdVeQlLEreE6lmUmGdkQaION3fwu+taMw/BHcMiKHMKT7C5xqStd1xRibpu0lTjqixd69pMM4+1LTyXCxlGuZNvykD5iGCOTSrxDfY67gLypBZ+R84XK7frs2OtdDovO8BriRFzG8y+ha6Nhixh+JrHu8zBLc/uDa2Qdg5XLZejkn/E/xztf0f7H9n/RoPe6Ay0s/758PHZ4MgafmP4H1ki9DP/lfG/gz7MOrb/PR6d9UaE/+2dHfE/v5b977kad9DXc4myPsFfSFQ2s4Pdt5tEoJsUfhYSjXiL1D856fnClOD8CeE5b06fvXrlofREyCEsEPVrcuuOQQoH9j4E5RrBxlmayP2qCDCBekkYXAUef0uTJSidbwMQ2vF3EcUoC/poSkMTCt4li6BAiQ00+Nu/xijlUGKS//2Tvi/e6A4K57vTV7Jk9fn2PwJoWC7DNC+DZnN4Wag+xAb0YytCUB2ij6BPHqlUoOJikiBG+REqQ58ofI0uNlfQ8g0IrjlkCUR/dLaFPyDi0/z7TVQqIFaUFFmUYyvQ2BRDL0oJsu61yG7/6wKUGjKUvgJFOkmisPdEOJ2K7DCI1LET6qnMO+7nWESoMN6gxIF8TWCXEwu0ZVtD1S0YzvDyLtPow8BZD7SiYgK/gDlVSD8sVnmQXer0QNAECCPRU77G+ZKUdczXmxfPvn7z7Qv8OToDyf1jcfo3XkKzQgUnA5Ua1NYon+v7jj23J9/maADJJejdMLOVQkD9IiLeRAUohT4ZqtFZXTIUziZaHibJHApIZDnqzpdZMr/pewJH9Q2M6fyNLF7Rky9fv5r/r/7832S0uiyLh9Xjkx885/HQlT6le39EyFqA/XxYScugKGU+x9bqgr6EW9jG1zi/w4cXRVPELkh39p6CQLM2OTZxGX0TBrF8k371NI5WyYmyiBVklLiPfP7zF18+/e7lt6jR1nlVXOD6SqQxpx4aI+cHLmpCdXpC/ZxfBOH1BRQwwVK0LeY5GmHioJDFWBnIdpYRmary8zSaX4IoXvig3c8zTQtowh6hnYOp/TAu5qBA5tKPkvlSBmhvgdb13b1aaBjsavaGwakU7IP5SKkdIYbjGmsML4EzARG9B2ZJN6WVp6/p9QyYP1raCuB2vTPcMHbGol8IBy3mFuZujaBI0T89EzcylGzMYXhVDnwPbaTIciu8GTQtM2tjpRcCUKCxNBy0TxYTx+mdeaA3O4M+/Tsb0r9e/5z+A+fxXNc7ER9yBUUGq2FORjusoeuPPNHzu0ADv+t6rvhUjDQxvuH9Do1Y/fMtSO1ksH6LneoNt73hWHe4DOIY3Wg2bWAnSIv2scefGfBwHPq99eTg3FkHGYE1i8m000W4aA8/+vgxQAAnjF4GA8j233NPFME6i6Nkxf2atEw6wwWKeQbshE0nvW5XJUQwG4La1migonK9+t1gS3fRjorz2iXeW8MG1XHAqzwqyAhDCeswosMWm7qpxhjKaSVhMXs2arLTM2iriUeE9DYgkQrw0GVm4ZpgO/7OBsyhdUuia2KKieFvRg4MWzBBgaxC/lIp38A8ZzkjYJkE5RF7q3YYiIcV/bXM0UKaLqMwCmIXlhfwPVigm1CibMZmfpTxsODnVCo5UAwSHKrKUTbC9qA8cwErFruWxhvGTWPlKD3IZAUiT0BeLhJ+irEW8yyrK0yGJRYHpIH2vf5XGL/bnwhH2fXOn0D58K/P3rcAGriRFyQXit6owuRWtnbCCFum2iaYszI93o1b3cOG0sDRh1szVH4wdNVyxDQtnoe8JuuHTESXvIQj22K7Bk63s5HNikTtdlB4oGbwfAnTBk2dUdBwaiLEmb2k5K1MPeWhBwkBJtIqWK8D9N2me64S4Sw3kgAA7O2tfGzGM6YdnV123UD7E/SIwFxc0z+yWnf9x5ZhvfKLcX7z69NPrcygIyALAU57jpxWu6seUAEthnrRykF5oA2O7asTp1yAKw60hhh/b4Q+0zsTPqF0boODhXGUsSW4S16wEwUzeI6uK62cOSRG+WqCBqA7wWOQRkqb/wh5JYF984EFR1u8NY8BJQlHUWt7DPOQtAYLvT8ltz+V0SJQAGAht7Cqy9RTFmxKi8yHyAmsIE8LUOJeQno6boIL00CHo2Jt1C2U/UHTq6lgEh0va1DEQClUbmvbiVNoVoiamEA/T6qkCXK2m+MswlltAuBOOWYvU2w7NjS4oG30DTBomqFqisKaUOeknULGS09TTDP1YEOsIrfWGqbzq2T0k9LSN50B5syBonD/n6rBELTBVHuCZ7M0d2a3MpaJaqTVFjVp0Jdea5Zr51zJEk8FmC5Gbm2l1eqEZtcKmkazZmVkM4hu0rnq0CLlyXp4e9Qkchs0co3f9u4S94YBBhCZiiLeP3v/NsOVZqaK3msLcfqFcBBqg8dP1OzXldEBKaNou6zqryKYwWr6hU2eCKvCNAQmeWrNYnsdsfQZA9uU6GYFoRC2Y0SdGM7IIhI6yfe3DvEI9N7RibV5QM7wWgPTC5f4v/rBuo1xqTldD1mO8b7bPjXIpdtejb6Cf1isEijqDN369FDtyNPySZcH5pqlI74Dm+c1ys4o69cQ6q1MeFTftEwFyF7HnlB4gTV9Vz/t1kDKxiYGcl0Ge6F70mwv0AgdkKtNulHbL255XvsjtWOGpKRYXkqCqawh15Bz1nZ4lEOuoL4rlEVkAuQFIRnkBGt9VScTKrDM+qoiE1U4vcItZqtcuB7C7dU3vBdsHQLf8H39i6UiPZcn4l0HlEaJsHjeGFDImJMhyKE6oLAO+fOrJHSSTvdvgRNxwg9gEpwNW9SfDkpArVWs9w9YvLd3tb0MOBRTVKdnwBgy0EucgYdd7LnVCjXQAJy0oCCWsi7sl5ssls5b2Cw/pYfITv7ANSUpqAkBzDnGcrHND3RsQhcqGA6IL5mnsXpqRU2q0xl1HmOEduQobZxm+sqrmLgxb+L9masPlIE8ldNWbJ5XgDBqmS9vgljJIIhOaiPbQZ5xgJ5+mTrYVS6VJGOqy5luZ+YUDC4Zp5iqAZ7h6uvO8GxJnTygRGUbx/W5MWrJQF9w1wC5Gm0UeAKJT5q0JQ5CxDpSI0iERWjBxGYdWuWbqpSKmOanmhPqqNbBMySHD1S1HaXiMhmYQm76DLagomK5ZkLc/jtjxRTiZoFIogVtETDB1AaKtglgymUagqTifJV+h7KvK25YbkEpKOWqzNgnQA5ahro+I8UCx+zvbfZT2JOrA5XQM9BEEndmn9mbavpQujVhWlThnA5bV1dNLOJF6YZUG0/wJ0wSlF9ZEaHsc5CTFHwDSmkz6TpYg6c0SrmYfBnATlUb4inuiTyLg1KzxqpUBBOq041VN/Xjig261YxA/dhMCUQMgQCDCifJs5vgJnqL1sCLKI4SGcQk+7LlHopEZTYqUssMg5s6UCzWImxfZNvqAN8Ba4Ol2d0Jz6OTUGgrOKjscadY7pgrjLCSiiwh6J9YLlkopwsjToHoN9EiWqQ/AyGpzAyuMj/8iWZrePuXONzEbHaj08lwIyoiyqiBvRo+PgS5B1goUD3PoxWlJzgSSEGMExKXAaJvJWPOeAR2pAbEG9j1THyaArbnIPZFj2bYOjU9+RzUohEuGdQpQFg0AhpCbOsnpu8StP7vojf/waCa0Or7kZqQ6CBQE541x6B3pjmvDSOzV/CLK5xfO5jFj1CZIiR3rg7ZSlCSgc0CCWBfiMlixbYwDeG3wHsW7rdNHDP8liDJBcqpgz2u2/OZ9amzojE6C1e+jFaIvbvEToXpjWPE9em2QHlttmcdAsGqvtXxZqlR0cjY4imOQ0+ePnEVCi8GFqzumO0qShSQvdqoeK8kUBjv2yDOnHliuUFNPMiVZ0ujFCddrzphACmfjPYkPhg18nl4CqxJ2fgGjRnlJKeIFYuh2gqDutxEShDxjAqZyVbNknUZwrgp1qMbPSYzQMsGy/t00oDxk42qMqpaHmyWP5Q7Qfuqpd6RLTtsWnAi6DT9UIBbdkK7zGDysXWQANhqyHxEJkxR7JhtOMXYWhbwF8moGCboFWMybeBmg+5q0DGLjcg22BTyASisMBlj+RCD0Uf1vmCGzeF12tg23M/RkY7WudufigiXyEbGSMOcDTPEwflkhl8dzMCx1Y0c7TdS+2SUC0dtDgXS/DJCiwze33CFhVtny9dSZiw3LCsZk+VQrdhOzNcppp7pJWqmRWXKq7JMcWGmqyhECHbus+3brI7pSsUBIIzvYWmP60LNuqoEPTaJqRwxDHqzpQnHIUPQNAa6fyjROm0fKRmLFW25sZ78lQJQhX4ISaC0dc+8HRkbR+ssspHLWhWtyXl1NZsM2uJ34kdVTd1wrKr+P5CmrpwfsDRbrWhYnBsDohLtW/BJtAr6qHWg6/+RUH2lYCGffir6J/d0ippWwBaEplVd2u8NbxMI5p6utVEXhKrqLNW9eGhrkUZoPCzJewxiz0Ixa9zosGzU0XG3qDLcX3i1tpplN3ZBaLNJe0+xluXfiMEtHoBfAElhEEXaI3cVzMNNsEB4CxqCGCoz6aFBHsgBa3/y5KzFNfaM86TMOBUaBwU8vWR0UYpfA0uMMRVLAw5SQG6B1xhMzoO2eAZW46TTthaOdlBpE6hJe8LZVY93zcfKNE1LFa1luPPLKKad25R1KqwabMvNqbDK7uHk5X6ak1pULiagfT9A8YoJ6Vo+Kg64oRrA0R1CPKFE2R+hMu3Zj3a1R/USdFV4o8t1qNSuZx7u9h/WDnrp+B9d6BS3eae/6rAQjMbSCHcCguEpyDybNATOsdU+HVIENwvI8kDdC+Xy2uGBSthW4CJPqC+uboURq7ihspgzgIR0LlxSlTWaf+1bpF/jwUbcMakAZPXa5Gzd0qKPW+HFqpBSZvYrGegZaK66HhLeSF7hdUHWZSV23ASwswck/2Qpnl6iI7lKWzNLFt0gC4lcIlC6j05iTNcyocUsa0vROHLJL3zYI1vj1Egke4+mNk9anGA9REn0/b5rW6jZrg1cpGYnVbOgheHUzM77Zm5rZWEB0z568Ohbd1ZLtAAGsdi1GsVPeQE9Er2uJ6rvvLwtj2/r4lJ1QWKsAVcS7np7y4yS9SjZbi9ZfQtgcjh3rTy3bX9w7IXIMxnLcr1qha5bgtbwguWFhqv0xTffum1bjOWofKMW0n2OyhYfHK8blMAbJjjjftsqT9KOtNJmapTXlN32lYcrDP4+3Je2fbATDc+FasfZdhrNDppodavb0zzcLaMqNC6YbcMls/P0UdwPcMWoQreWG2Y33fPBqEOz5KPe3u3Whnm4//S063eHGAWpix6VrjEeWCPg7Jm9W102W/eQB2FPWH1YgTtd4EFHAvPS3DE8lWxCbduD7VyYMCIAfRSTQZ/OljI7r/aONxpOXEcrY4iFPR5+I+m4P84lbbo0AGrGZWjk/4k+KZ+z/bU6H2lrJ6Z0dk1gAzaJit6AslmAm12itO4sKmRSOTKXJMK2M31t51WWad0F28eh3W3TBl932zcUXVJY2J7RaQtLc6voXFTHzL3b1Zrl6UXjLKhlXUeOjzAyqNjlcbQXf5ufBpNOo3GErBiSz1xcSLXJCjO13UPD6uZrBmzRMAwxZhqCMW7SuJQMl/z226dVclSmuzVVcBGtC4rn5RDIkUGN6lMM3IY+t41w3QNdYE1jTtqLqQjah7f18GKR6XARrdZptKiWg7ONXO39qW9Y2MQMqOFk0cF6MAiASY8DYjatDIZq6LJXqu5VcmvDR1a4JAxADUdHLN12D0grnrV7kpCJMeqQZuhQrC9mmkIe12KRDlSwm+BuW2tWiZ7bHmoFdOwXtYBuq+2/ZgyxVhhV9TDQ10Et8WOYgAVzDIxAqqA7BGDAH3QqgqVG8g4dYCpj5kcs0JLdw6ogSsJ4E+WWycyCrYImh8BVqiHhqCBozDLZrdPf9VgV1HlrO1uooQbGBhmqg8TT5pnh34nqIXnhZm6TsLiuocCfT1VrEkQFyHFWL7BcK6zqOovZX1aBCyw/UN3Go1PvHYKm6toEO53DtRkzsv49hoySmrZpkYHTGiO9kgvF64PEMwZWhxwyrDPUEC91t16rDW6KkU1tt+XdtjhI7c6OBxaP53+P53//lvO/w7PHZwO/j9+Hw+Ny+o3F/zPRbn7hE8B3n//tjR4POf7/GQb+6w3w/O9wdHY8//srnf99HRSp6JHMVuagVCYK62IZ+RABljJONS1SRl8sI3QCaiSJmTygQ76w8rLvPU+TBUbF117UMiV8CYkGNwwgrw74fsGOPvZ5Vviarnfu4lFjqu+kCuID0gbjt9HmrwVQJ64Lodzkha0Ze1pzDUiN1SeaPcLeoRx0QuIFgZAxkekvSPkrxPx65HpFqWchycFnhFwVuJqiJD71ySorC2ppcIXzLSEXdSYJg1Kc2JQXfDgCqvmfFH2fm/PH198JZ5GGxaPAOih8qjP564V78Dzv3/8IL6fwzYFdjiPauFudGijEc31+97kP/0++fPP01b88/QbNk12/h6aePn4M8IMsPyP8OMOPx/hxTh908wl98LG5nrHIo+euflzleYvvvTqRQp5sr3LGK3u4BZEyBgw7So8OmNqI1tMG5HpwDDll0Lo7jlzrCSKNaiM/V4V0JCprdF8NYHNvBPiFVIVxujp2soq2rXWSh4TaNseKcDGhOT6XMXworZmwKOt5dUjvDtfcTsfD6qqYWOR20q6yU4Egqa5bh0APPKGdbo1Utk9vwIBH1Eix2LpGxXCHhNAUtWhsJtIZZaxHY+tiYLcfH5Z4D2cm40ZEL8qj7BLELNoScRMVVlcu5jqlAQ8pSBrdd11GSsBXn05WKm9K1x5wen8EDtHEjg6PNxTYHV3pE7Q7Kz87egG2tcRIZXKvu+1W13BXS76rJ1eR4LEGZRpxVZx4vGWMJHul0pquFUzzrXL2q/anRTgHqtRSVoTiAVFp9+rAvNmomdWK0KkRgSNdGw+QymFGqL10c7JuTpjMe1qoIGBWmT15OjjQcEIMqAKf+3VEwZxP4AyMqxO2b+BZwOWUkxJW6wqt9VKbr3UMBM/s4BPi6AoUDjkmxM3R2zRX2yWiyjTevgENs96xo3BfCKav5FTjKCVbG54sN+8qtIUO9foJtMYFuUaq6irHMO8TzoZu4AOw/TvB+lQB+gREyvuKCh5BACuC1KpDdf/765dfu5+TuAHkxET0MhAregd0VbUvtGJPGhwcrkFthyMW6olPgLN/AsKBNq2ApIO3kNafiMrQqmFoil7jyt2F+AV2BOLnTJ/6w9PALEy5ImXMCYg79YFrYmetwCF3BcWrrJ08ez70cLXe16p9nm/RDLkbS/sgRLUFYH3uNxHVesLnDHs+4JSjNy+xzRsDYIeQzOElIpPgIgYug78QtN4JN4ugY5kTYVMM6e1G6ggK9o/M3uoGM+fnvklAi9EcUKkifyqgIsrY1mpwSCoFYdpELFVirh3+BwHT+g0sZuUoSy9icMjvwmQaEJVsbCm0v44oPb8DUfo3k0s1wHK0VUwpHzT8ahUfcmu4QlvnOLEGYc1YQvhhgQrXap5NUa4i8NSWTZP0bpT1zC55GrEFU704xTryRdMHx3XLVvqZqdczNTSr98SUHtUMo1XD/hYq8siTb+0AMdctfsqKnp7lNXQNA2MOzX4yr+ZMAYqE8Een73LjRKl671VNspq5pNN8LDuuGsIjlmb5g5bTDs0yKKYzU8Cnda4kEzsVMEs808/fTHIjR+MBQPMjt8/hQfe0LX1Zf9GOxSTHbSiM537Dfblqg4O0YZnMtv0QvnoI02S3j6KM9A/wQH1d3BOUoIGe2oODYBWjkWnAnhByR4sMduSkDjMPDe0rty7shhczw755o7P9vuSO3W8cBbQNlUuxapx9SFbvkaaNtV/25tQc/yqYUE35qhaHEl/ygVa6NhS3GJ0aKhGomTdyYmanxVitk4cE9iJ+jVDqSnggpp3LLMJDx+pwGedHQJmSVxoQ46a3ntqx5xUy9Onavig8nZOYGPEF9M6Znm5R1UxwYRk2SWXCsrsGRXjSgfEB/tRxa0fSKvUT0wLPmxIl7JcDUm0K6p1uPvB8GnnoKSCHUrk22RwK5FLKPNo4WKYmF/ndkVN3TWrbM0cv18J21jxzkGjKbcQdncuv6VSbTA07EiCYL2HdXgVq5OcgTs0rRGAATa3daswL6xxbGGHMBUYIUomCozSYCHrh5vYvKjKQ5cnDGtjdSycPgxiaEUpLHrXrH4t3+svUTLfZ+8/3W2qnnKoX4VWRIWbqvZosSz6rB+jGN5LQkXk+lEcGGZWvYbRrRrXxKfR5fR0oyvNaODT5zQlMkIAoDgS/wpEsosZIp4E0uT4uuEl0mBwNmHGQtIU1BnoMYYcuA56MhuJA69XtfyVWz4n4b7lw0yBPTyrFIcz5isXeiQk1k+pLQhna9MlamiVFcxarnGQdcdXZz1K/8e6uZLC1BnOzDO9KiXHUAl7lqp8X/CuZ87LqGkEB5znOPMMzrIm1/75Ozn2AhVZbCFfzGQsF8LUpFXCNwIK2zLBA6tEiiioQMrmaPS5mzUhCFeva1niWxfksxz2dhiZ+iWM0Pcgs9+B+BKnD3U0GIPMhr5ULTW1tTSuiBfIIDEvXcfn9fAqcZA2LhSTkoPY0OPRm1MZaniazukRjhvKziYI84S+3eSoE6YTEGbfaNmr4CBNjPmhN/0A+L4Mmp+eGNQQJNo9dUeA3yE1GwJ7bYiWpZqmaNo5O3lUncN3mLNo/FNSssbZMeCuqExd6uVj4QenQEqSwGb1DSfR6plRcA7frQHq9XCm9OblqZVSloIFUHRUljnYI87a31Gtnb6gDsIfXiUElakGODIxcwmTZeafKwjx+f/n+9B3abM1NNNTSyzbpYcezWD2ZA3V998bk02SjXPoHZRRZWM7VnZrJDU8+fyqstOKRqOprtWzuzeE5gfVqxeoB4QJxallVYLFsi6ufBUUCemYVevZM9ZjPnRzxH0f8x8PxH0+G/vng7MmT7vkR//Hbwn/s0vjv8PLH+/Efg96wV+E/Roj/GAz6vSP+41fCf7why/BoXLno0U/R650WciWc7zAIUbwroxC2dQy/t1kFeT0UGr71kbH/kiO184sZ0fpciMoKIQ/CQpyaYsexO5TLJCC4sWfcQCcqeDz6ObI8pVillpaHlVdH9NVrJVXl2T7ORZm4c4nyI75PB6OMWAgRPK4drTcx2cQJQnKTqvdkaVxGDXeRfvALJj8UgvFBQdIZgWGRx2AxWD1RCWrPEJBh3n6E1Mnn9NxRHjeM0+cBAT18hxRG+42jRbAX/ZwihOYZQnEwFLEaCJxXYwLTGK8enoDEVwGYeHsmsD6F9Ccvl2Q9vUJmZ8q5pmeMEzeiRRba5kUTghT9JF1f4Is5ZRIHb+mMCrYDsUXkoeOSXA5FqV5Tr5DqamKrYDQ77Dd0WU0CPMxPIPadZFRPmNJZe6tQeg3B/osDqjfaQ1I1y7mR6iBGBxZelHSQ2HgOowP1dojq9mmMFIEL1/g21MIBoZmH4/0jqqt49I7KRfFYbqFP8/SajxHdl53jvt2fnTR0bK1qeF26pzfKTrCOLCgvYebGEU47J3GbxyiRJmQ2PNiFR++wtPd+lqw6bcqlroQaWjhcYEsUAnZJWqGeoUaa2e8fEbn5c64rT953PjgOdFudfhHcSN2qugqEEwWXVVLoAMWgZO+bNowCR5RmUhFJ5u9ghPaSLWQBanV6N0m5rBaiHiasKrelYWo6Fbt1HCXXjm4eFe7p9uwTh5xWeiTa51/V0HJbYtzxHzouMsjlZXsrrGjPSTE+qItldTvCXpxnCqHDL2gDfQz54kF9jiMBZXQe52x8p/a3vPR/yGF0nU5XdECL7YiOf5VG1P2bsT8CZZY6cEPLShl5MnXwEfFJnT8nHctr2iQdA3h3wTo+QCnTgGUHB3Ys3ukhDi4K/O9wUe77Pye0GBT+YE4/cAj+nNwgRJTvwle+RzHix39OhOiOK7uyaazhcsTcDG8jrlYPL0shZZUxi7ItydRf+FghcKo7+JzL0Ti4O1GhJ+Kh+Q+0tiwu7TOROqly3DP/qqGFXutB9dtqrbqF/a2PqtbsW4fUIPNUTGcdn7IdeFNtxl9aGJd7IlRaUBfyZOzhXZSx3Rwa1UAEK473sg5fsHxHJpqkr14FYfnvICla6waeOEW4F5Lm7YT4LMHOdM+wt8BLAnp5RMGIgruWHL4vAJozGXRhDQEjv0gLqRzV9j6ENkd6J0KBgVYI8ELHZ/nIV+5TtNQG32u6wtAXZm0M5Ajyap6yQzHfVdV83r12YLERzD2kxNQY3JiW9cR3HRJvcc2dhm4Td7JWDisEHVjutQfF4jwUj7MKxmn7vtQM0ZBQkKfneNS1jPYM9+MT2yqveINKcS13E+A1F+g3GovTUFvPD/kSawb5sGGQb7VUm2PJBsFgfH/kJfyZPkdtZK27FXVlIbfvPs8DlJuzzd5KyRU2XBT6ZdnAzdih+TAno3Z+7jmhG4HgUOCd4zvgk1StrSrMyb9qtYygR3XVL1CuGtF0XI/rJwJ0O9RbvupRU5WL0vjeDEwu+x40FPgrYQNd4l+iYPtVaz0qDf1odJKQBHf9LvtFLaoJUsfwuBJnwJI+EuOwQgsuvd+ef3IFBeHaPXzBCXwO6LPXxZhV6n3tVd2WR2BAcWbcn+kHC9W0Q7jMHevKDpGBqyTXk84gcnRBNJPM2qoGhOJQwX2aDfDg95NqYtRYqwqSgbO+V+eh90aWLRDlQx/VGsKirPCy+qIwsx8eYLZBhr3XdkzhAYbcpvDdSi6rB6Hl8LOzlviztcizP9fLppYxtk/tSYfcY8tMe+KQRPuPk7scdeoVyexQ3vNPcYbmweM7vFz3j+qHuOkuo5Ji0tW3w7ZmQ0qbpdYmm3H7mgxtbjuzmD/TiPn1FEqdHUrLZCW34I+mYPdg6toYiVPKqBvda82GrygzLIsWnUWsfdIj8q3xShu8dXDQKxb0GY3ZJUi96SoP1g6VNK06xW+/QbZmhCeWKpCyODIwjkvra2IFUyBxL/t+QpR9RPkI15owSKqLb74qzGMoEP2GmfWwzCbMySfMzCfLxJxer8gztk+7Tzumb3OMBlvMe9fqrVX2o971fNB2HwhID4iWmQMD5VXEOvDOHLWc8cQNavP6zd9mP3yn7EJjMVUvvQDBfPYexXCDj6lHdZnZ5qQZHwnLcF3RfSO6N6NNT/c52cPfcWTw0W5dn57tWS9s04XekUCr28jCOZ7JP17H63gdr+N1vI7X8Tpex+t4Ha/jdbyO1/E6XsfreB2v43W8jtfxOl7H63gdr+N1vI7X8fp7X/8NXwRIEQDIAAA=')), mode='r:gz').extractall('.')
print(sorted(os.listdir('filamentos')))

## Datos, separación estudio / examen y refinador de la ronda 3
Misma separación que las rondas 1-5.

In [ ]:
import glob, hashlib, json, shutil, time
import numpy as np, pandas as pd, torch
import segmentation_models_pytorch as smp
from PIL import Image
from ultralytics import YOLO
from filamentos.datos import cargar_anotaciones
from filamentos import candidatos as C, yolo as Y

disp = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()

def buscar(patron):
    for raiz in ('/kaggle/input', os.environ.get('MAGFILO', '.'), os.environ.get('PESOS', '.')):
        r = glob.glob(f'{raiz}/**/{patron}', recursive=True)
        if r:
            return r[0]
    return None

json_path = buscar('MAGFiLO_1.0_Annotations_kaggle2026_train.json')
assert json_path, 'No encuentro los datos: añade la competición como Input del notebook'
DATOS = os.path.dirname(os.path.dirname(json_path))
PESOS_REF = buscar('refinador.pt')
if not PRUEBA_RAPIDA:
    assert PESOS_REF, ('No encuentro refinador.pt: añade como Input el notebook filamentos-detector-ronda-2 '
                       '(Version 4). Hay en /kaggle/input: ' + str(glob.glob('/kaggle/input/*/*')))
print('Datos en', DATOS, '| refinador:', PESOS_REF)

ANOT = cargar_anotaciones(DATOS)
nombres = sorted(n for n in ANOT if os.path.exists(f'{DATOS}/train/train_images/{n}'))

def es_val(n):
    return int(hashlib.md5(n.encode()).hexdigest(), 16) % 100 < 15
val = [n for n in nombres if es_val(n)]
tr = [n for n in nombres if not es_val(n)]
if PRUEBA_RAPIDA:
    tr, val = tr[:8], val[:3]
    FRACCIONES, RES, MODELO, EPOCAS, HORAS, LOTE = (0.5,), 256, 'yolo11n-seg.pt', 1, 0.02, 4
# Orden fijo al azar: cada fracción es el principio de la lista, así el 25 % está dentro del 50 %, etc.
orden = list(np.random.default_rng(SEMILLA).permutation(tr))
print(len(tr), 'imágenes de estudio,', len(val), 'de examen')

ref = smp.Unet('resnet18', encoder_weights=None, in_channels=3, classes=1).to(disp)
if PESOS_REF:
    ref.load_state_dict(torch.load(PESOS_REF, map_location=disp))
reales_val = {n: Y.reales_rle(ANOT[n]) for n in val}

## Los pasos de la curva
En cada paso: entrenar YOLO con esa parte, sacar los candidatos del examen y medir con el juez de la ronda 5
(confianza >= umbral, unir los que se tocan). Se guarda `candidatos_curva_<pct>.json` de cada paso.

In [ ]:
resumen = []
for frac in FRACCIONES:
    pct = int(round(100 * frac))
    parte = sorted(orden[:max(1, int(round(frac * len(orden))))])
    print(f'\n########## Paso {pct} %: {len(parte)} imágenes de estudio ##########', flush=True)
    t = time.time()
    carpeta = f'/tmp/yolo_datos_{pct}'
    yaml = Y.preparar_datos(DATOS, ANOT, parte, val, carpeta, res=RES)
    modelo = YOLO(MODELO)
    modelo.train(data=yaml, imgsz=RES, epochs=EPOCAS, time=HORAS, patience=PACIENCIA, batch=LOTE, single_cls=True,
                 fliplr=0.5, flipud=0.5, degrees=0, hsv_h=0, hsv_s=0, hsv_v=0.3, mosaic=1.0, close_mosaic=10,
                 overlap_mask=False, cos_lr=True, project='/tmp/yolo_runs', name=f'c{pct}', exist_ok=True,
                 plots=False, workers=2, device=0 if disp == 'cuda' else 'cpu', verbose=False, seed=SEMILLA)
    hist = pd.read_csv(f'/tmp/yolo_runs/c{pct}/results.csv')
    hist.to_csv(f'yolo_historial_{pct}.csv', index=False)
    min_ent = (time.time() - t) / 60
    print(f'Entrenamiento: {min_ent:.0f} min, {len(hist)} vueltas', flush=True)

    det = Y.detector(YOLO(f'/tmp/yolo_runs/c{pct}/weights/best.pt'), RES, CONF_MIN)
    cand = {}
    for n in val:
        grande = np.asarray(Image.open(f'{DATOS}/train/train_images/{n}').convert('L'), np.uint8)
        cand[n], _, _ = C.procesar(None, ref, grande, disp, RES, CONF_MIN, UMBRAL_REF, 0, detectar=det)
    with open(f'candidatos_curva_{pct}.json', 'w') as fh:
        json.dump(cand, fh, separators=(',', ':'))
    filas = []
    for u in UMBRALES:
        r = Y.nota_umbral(cand, reales_val, u, 'rle_ref', por_tamano=True)
        filas.append(dict(umbral=u, PQ=round(r['pq'], 4), SQ=round(r['sq'], 3), aciertos=r['tp'], inventados=r['fp'],
                          escapados=r['fn'], esc_menos_1k=r['escapados_menos_1k'], esc_1k_3k=r['escapados_1k_3k'],
                          esc_mas_3k=r['escapados_mas_3k']))
    tabla = pd.DataFrame(filas)
    print(f'\n=== Paso {pct} %: examen, YOLO + refinador ===')
    print(tabla.to_string(index=False), flush=True)
    mejor = tabla.loc[tabla.PQ.idxmax()]
    resumen.append(dict(paso=f'{pct} %', imagenes=len(parte), vueltas=len(hist), minutos=round(min_ent),
                        umbral=mejor.umbral, PQ=mejor.PQ, aciertos=int(mejor.aciertos),
                        inventados=int(mejor.inventados), escapados=int(mejor.escapados),
                        esc_menos_1k=int(mejor.esc_menos_1k)))
    pd.DataFrame(resumen).to_csv('curva_resumen.csv', index=False)

## La curva
El 100 % es la ronda 5 (mismo sistema, medido en el mismo examen).

In [ ]:
resumen.append(dict(paso='100 % (ronda 5)', imagenes=len(tr), vueltas=88, minutos=153, umbral=0.2, PQ=0.4532,
                    aciertos=922, inventados=481, escapados=445, esc_menos_1k=236))
print(pd.DataFrame(resumen).to_string(index=False))
print(f'Tiempo total: {(time.time() - T0) / 60:.0f} min')